# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


# **Load Data**

In [5]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [6]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
from Recommenders.BaseMatrixFactorizationRecommender import BaseSVDRecommender
from Utils.seconds_to_biggest_unit import seconds_to_biggest_unit
from sklearn.utils.extmath import randomized_svd
import time

class PureSVDRecommender(BaseSVDRecommender):
    """ PureSVDRecommender
    Formulation with user latent factors and item latent factors

    As described in Section 3.3.1 of the following article:
    Paolo Cremonesi, Yehuda Koren, and Roberto Turrin. 2010.
    Performance of recommender algorithms on top-n recommendation tasks.
    In Proceedings of the fourth ACM conference on Recommender systems (RecSys ’10).
    Association for Computing Machinery, New York, NY, USA, 39–46.
    DOI:https://doi.org/10.1145/1864708.1864721
    """


    RECOMMENDER_NAME = "PureSVDRecommender"

    def __init__(self, URM_train, verbose = True):
        super(PureSVDRecommender, self).__init__(URM_train, verbose = verbose)


    def fit(self, num_factors=100, random_seed = None):

        start_time = time.time()
        self._print("Computing SVD decomposition...")

        U, Sigma, VT = randomized_svd(self.URM_train,
                                      n_components=num_factors,
                                      #n_iter=5,
                                      random_state = random_seed)

        self.USER_factors = U
        self.ITEM_factors = VT.T
        self.Sigma = Sigma

        new_time_value, new_time_unit = seconds_to_biggest_unit(time.time()-start_time)
        self._print("Computing SVD decomposition... done in {:.2f} {}".format( new_time_value, new_time_unit))


In [ ]:
# Define objective function for hyperparameter tuning
SEED = 1234
STUDY_NAME = PureSVDRecommender.RECOMMENDER_NAME

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = PureSVDRecommender(URM_train)
    recommender_instance.fit(
        num_factors=optuna_trial.suggest_int("num_factors", 50, 1000),
        random_seed=SEED
    )

    return evaluate_recommender(recommender_instance, at=20)

In [ ]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    objective_function,
    study_name=STUDY_NAME,
    n_trials=100
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [ ]:
bp = optuna_study.best_trial.params

def refined_objective(trial):    
    recommender_instance = GlobalEffects(URM_train)
    recommender_instance.fit(
        lambda_user=trial.suggest_int("lambda_user", max(0, bp["lambda_user"]-100), bp["lambda_user"]+100),
        lambda_item=trial.suggest_int("lambda_item", max(0, bp["lambda_item"]-100), bp["lambda_item"]+100)
    )

    return evaluate_recommender(recommender_instance, at=20)

In [ ]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    refined_objective,
    study_name=STUDY_NAME+"_refined",
    n_trials=20
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- ADD HERE

# **Train Model with best hyperparameter**

In [ ]:
# Train final model on train + validation with best hyperparameters
recommender = GlobalEffects(URM_train + URM_validation)
recommender.fit(
    lambda_user=optuna_study.best_trial.params["lambda_user"],
    lambda_item=optuna_study.best_trial.params["lambda_item"]
)

# Save the trained model
recommender.save_model(paths.MODEL_DIR)

In [ ]:
import pandas as pd

# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = recommender.recommend(ids, cutoff=20)

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, STUDY_NAME + ".csv"), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")